# Environment Setup & Dependencies


In [ ]:
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_dataset


from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)


# Path Configurations

In [ ]:
OUTPUT_DIR      = "/content/drive/MyDrive/Kazakh_OCR_Project/trocr-kazakh-finetuned"
LOGGING_DIR     = "/content/drive/MyDrive/Kazakh_OCR_Project./logs"
CSV_PATH = "/content/drive/MyDrive/Kazakh_OCR_Project/dataset_mapping.csv"
IMG_DIR  = "/content/drive/MyDrive/Kazakh_OCR_Project/images" #this is the second version of images dataset with multiple words in one image

# Training Hyperparameter

In [ ]:

# Training hyperparameters
BATCH_SIZE      =  16         # reduce to 4 if OOM
GRAD_ACCUM      = 2          # effective batch = BATCH_SIZE * GRAD_ACCUM
EPOCHS          = 10
LR              = 5e-5
WARMUP_STEPS    = 200
MAX_TARGET_LEN  = 128        # max tokens in a label sequence
FP16            = torch.cuda.is_available()
SEED            = 42


# Evaluation Dependencies & Metrics Setup

In [ ]:
# ── optional but strongly recommended ──────────────────────────────────────────
try:
    import jiwer                          # pip install jiwer
    HAS_JIWER = True
except ImportError:
    HAS_JIWER = False
    print("[WARN] jiwer not installed – falling back to manual CER/WER")

try:
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
    import nltk
    nltk.download("punkt", quiet=True)
    HAS_NLTK = True
except ImportError:
    HAS_NLTK = False
    print("[WARN] nltk not installed – BLEU will be skipped")

import editdistance                        # pip install editdistance

[WARN] jiwer not installed – falling back to manual CER/WER


# Hardware Acceleration

In [ ]:
# ── device ─────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


# Model & Processor Initialization

In [ ]:
# ── model & processor ──────────────────────────────────────────────────────────
print("Loading model …")
processor = TrOCRProcessor.from_pretrained("kazars24/trocr-base-handwritten-ru")
model = VisionEncoderDecoderModel.from_pretrained(
    "kazars24/trocr-base-handwritten-ru"
).to(device)

# Explicitly resize token embeddings to match the processor's tokenizer vocabulary size
# This helps ensure the decoder's output layer ('output_projection.weight') is correctly sized
# for the fine-tuning task, potentially resolving 'missing keys' warnings
model.decoder.resize_token_embeddings(len(processor.tokenizer))

model.eval()

Loading model …


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/957 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/273 [00:00<?, ?B/s]

VisionEncoderDecoderModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (i

# Custom Dataset Definition

In [ ]:
# ── defining custom dataset ──────────────────────────────────────────────────────────
from re import split
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset

class OCRDataset(Dataset):
    def __init__(self,
                 img_dir = "/content/drive/MyDrive/Kazakh_OCR_Dataset/images(2)",
                 csv_file ="/content/drive/MyDrive/Kazakh_OCR_Dataset/dataset_mapping.csv",
                 transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx, 0]
        text = self.df.iloc[idx, 1]

        img_path = f"{self.img_dir}/{img_name}"
        try:
            image = Image.open(img_path).convert("RGB")
        except OSError as e:
            print(f"Warning: Could not open image {img_path}. Skipping this sample. Error: {e}")
            return None # Indicate a problematic sample

        if self.transform:
            image = self.transform(image)

        # Return a dictionary for easier batching with DataLoader
        return {"image": image, "text": text, "file_name": img_name}

# Google Drive

In [ ]:
# ── Mount drive ──────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Dataset Splitting & Initialization

In [ ]:
import copy
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split


# --- 1. Initialize Datasets ---
# We create two base instances to allow for different transforms later
train_ds = OCRDataset(csv_file=CSV_PATH, img_dir=IMG_DIR, transform=None)
test_ds  = OCRDataset(csv_file=CSV_PATH, img_dir=IMG_DIR, transform=None)

# # --- 2. Split Logic ---
# # Extract the 'train' and 'test' rows based on the 3rd column (index 2)
full_train_df = train_ds.df[train_ds.df.iloc[:, 2] == 'train'].reset_index(drop=True)
full_test_df  = test_ds.df[test_ds.df.iloc[:, 2] == 'test'].reset_index(drop=True)


# # --- 3. Assign df ---
train_ds.df = full_train_df.reset_index(drop=True)
test_ds.df  = full_test_df.reset_index(drop=True)

# # --- 4. Final Verification ---
print("══ Dataset Split Summary ══")
print(f"Training Samples:   {len(train_ds)}")
print(f"Testing Samples:    {len(test_ds)}")
print("═══════════════════════════")


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Kazakh_OCR_Project/dataset_mapping.csv'

# Training args definition

In [ ]:
# ── Define training args ──────────────────────────────────────────────────────────

import os
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    warmup_steps                = WARMUP_STEPS,
    disable_tqdm=False,
    fp16                        = FP16,
    predict_with_generate       = True,      # needed for Seq2Seq metrics
    generation_max_length       = MAX_TARGET_LEN,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    report_to                   ="tensorboard",
    logging_steps               = 50,
    logging_strategy            ="steps",
    save_steps=100,              # Save a checkpoint every 100 steps
    eval_steps=100,              # Usually good to evaluate when you save
    load_best_model_at_end      = True,
    metric_for_best_model       = "cer",
    greater_is_better           = False,     # lower CER is better
    save_total_limit            = 2,
    dataloader_num_workers      = 2,
    seed                        = SEED,
)

# Metrics Helpers

In [ ]:
# ── metrics ────────────────────────────────────────────────────────────────────

def cer(ref: str, hyp: str) -> float:
    ref_c = list(ref.replace(" ", ""))
    hyp_c = list(hyp.replace(" ", ""))
    if not ref_c:
        return 0.0 if not hyp_c else 1.0
    return editdistance.eval(ref_c, hyp_c) / len(ref_c)


def compute_metrics(pred):
    label_ids = pred.label_ids
    pred_ids = pred.predictions

    # 1. Replace -100 in labels with pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # 2. IMPORTANT: Also replace -100 in predictions (sometimes generated by the model)
    # and clamp values to stay within the valid vocabulary range
    vocab_size = len(processor.tokenizer)
    pred_ids[pred_ids == -100] = processor.tokenizer.pad_token_id

    # This prevents the "out of range integral type conversion" error
    pred_ids = np.clip(pred_ids, 0, vocab_size - 1)
    label_ids = np.clip(label_ids, 0, vocab_size - 1)

    # 3. Decode
    pred_strs = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_strs = processor.batch_decode(label_ids, skip_special_tokens=True)

    # Normalise
    pred_strs = [" ".join(s.lower().split()) for s in pred_strs]
    label_strs = [" ".join(s.lower().split()) for s in label_strs]

    avg_cer = np.mean([cer(r, h) for r, h in zip(label_strs, pred_strs)])
    exact = np.mean([r == h for r, h in zip(label_strs, pred_strs)])

    return {"cer": avg_cer, "exact_match": exact}

# Data Collation & Preprocessing

In [ ]:
def collate_fn(batch):
    batch = [item for item in batch if item is not None]
    if not batch: return {}

    images = [item["image"] for item in batch]
    texts = [item["text"] for item in batch]

    # Process images
    inputs = processor(images=images, return_tensors="pt")

    # Process text separately to handle padding masks (-100)
    labels = processor.tokenizer(
        texts,
        padding="max_length",
        max_length=MAX_TARGET_LEN,
        truncation=True
    ).input_ids

    # Replace padding token id with -100 so it's ignored by the loss function
    labels = [
        [(l if l != processor.tokenizer.pad_token_id else -100) for l in label]
        for label in labels
    ]

    inputs["labels"] = torch.tensor(labels)
    return inputs



# Trainer Initialization

In [ ]:
# 3. Pass them to the Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds, # Your custom class instance
    eval_dataset=test_ds,
    data_collator=collate_fn,     # How to turn your custom data into batches
    compute_metrics=compute_metrics,
)

# Checkpoint Recovery Logic

In [ ]:
# ── Get last checkpoint - in case if you already run the training  ──────────────────────────────────────────────────────────
import os
def get_latest_checkpoint(output_dir):
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith('checkpoint-')]
    if not checkpoints:
        return None
    latest_checkpoint = max(checkpoints, key=lambda x: int(x.split('-')[1]))
    return os.path.join(output_dir, latest_checkpoint)

latest_checkpoint_dir = get_latest_checkpoint(OUTPUT_DIR)
if latest_checkpoint_dir:
    print(f"Found latest checkpoint: {latest_checkpoint_dir}")
else:
    print("No checkpoints found. Training will start from scratch.")

Found latest checkpoint: /content/drive/MyDrive/Kazakh_OCR_Dataset/trocr-kazakh-finetuned(2)/checkpoint-900


# Training

In [ ]:
# ── run ────────────────────────────────────────────────────────────────────────

print("\n" + "═"*30)
print("  STARTING KAZAKH OCR FINE-TUNING")
print("═"*30 + "\n")

# --- Fix: Ensure 'image' column is not removed by the Trainer ---
# The Trainer by default removes columns it deems 'unused', leading to KeyError in collate_fn.
training_args.remove_unused_columns = False

# Run the training
train_result = trainer.train(resume_from_checkpoint = latest_checkpoint_dir)

# Log basic training stats
print("\n" + "═"*30)
print("  TRAINING COMPLETE")
print(f"  Total Runtime: {train_result.metrics['train_runtime']:.2f}s")
print(f"  Final Loss:    {train_result.metrics['train_loss']:.4f}")
print("═"*30)

print("\nSaving best model & processor …")
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved to: {OUTPUT_DIR}")


══════════════════════════════
  STARTING KAZAKH OCR FINE-TUNING
══════════════════════════════


There were missing keys in the checkpoint model loaded: ['decoder.output_projection.weight'].


Epoch,Training Loss,Validation Loss,Cer,Exact Match
9,0.010422,0.067522,0.035342,0.510000
10,0.004260,0.065755,0.035844,0.507500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['decoder.output_projection.weight'].



══════════════════════════════
  TRAINING COMPLETE
  Total Runtime: 3806.67s
  Final Loss:    0.0017
══════════════════════════════

Saving best model & processor …


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to: /content/drive/MyDrive/Kazakh_OCR_Dataset/trocr-kazakh-finetuned(2)


In [ ]:
# ── final eval on val set ──────────────────────────────────────────────────────

# print("\nRunning final evaluation on validation set …")
# # This evaluates the BEST model (because load_best_model_at_end=True)
# metrics = trainer.evaluate()

# print("\n" + "╔" + "═"*30 + "╗")
# print("║      FINAL VAL METRICS       ║")
# print("╠" + "═"*30 + "╣")
# print(f"║ CER:          {metrics['eval_cer']:>14.4f} ║")
# print(f"║ Exact Match:  {metrics['eval_exact_match']:>14.4f} ║")
# print(f"║ Eval Loss:    {metrics['eval_loss']:>14.4f} ║")
# print("╚" + "═"*30 + "╝")

# Loading fine-tuned model and testing on one sample

In [ ]:
# ── Load the model ──────────────────────────────────────────────────────────

print(f"Загрузка модели из: {OUTPUT_DIR}")
loaded_processor = TrOCRProcessor.from_pretrained(OUTPUT_DIR, local_files_only=True)
loaded_model = VisionEncoderDecoderModel.from_pretrained(OUTPUT_DIR, local_files_only=True).to(device)
loaded_model.eval()

Загрузка модели из: /content/drive/MyDrive/Kazakh_OCR_Project/trocr-kazakh-finetuned


Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

VisionEncoderDecoderModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (i

In [ ]:
# ── Test one sample ──────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
from PIL import Image

example_true_text = 'text'

# Загрузка и предобработка изображения
image = Image.open('zheke.jpg').convert("RGB")
pixel_values = loaded_processor(images=image, return_tensors="pt").pixel_values
pixel_values = pixel_values.to(device);

# Выполнение инференса
with torch.no_grad():
    generated_ids = loaded_model.generate(pixel_values)

# Декодирование предсказанных идентификаторов в текст
generated_text = loaded_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

# Отображение результата
plt.imshow(image)
plt.title(f"Предсказанный текст: {generated_text}")
plt.axis('off')
plt.show()

print(f"Предсказанный текст: {generated_text}")

# Evaluating model BEFORE fine-tuning on the test set and calculating metrics(CER and Exact Match)

In [ ]:
from torch.utils.data import DataLoader

all_preds   = []   # decoded strings
all_targets = []   # ground-truth strings
all_fnames  = []   # filenames (useful for error analysis)

print(f"Running inference on {len(test_ds)} test samples …")


Running inference on 800 test samples …


In [ ]:
# ── Define test loader ──────────────────────────────────────────────────────────

test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [ ]:
with torch.no_grad():
    for batch in tqdm(test_loader):
        # Extract pixel_values, texts, and fnames from the collated batch
        pixel_values = batch["pixel_values"].to(model.device)
        labels = batch["labels"]
        # Assuming original OCRDataset's __getitem__ returns 'file_name'
        # and collate_fn preserves it, or we need to pass filenames separately.
        # For now, let's assume `texts` in the loop corresponds to `labels` after decode.
        # And filenames need to be explicitly passed if collate_fn doesn't include them.
        # Since the original collate_fn doesn't explicitly return fnames,
        # we will extract from test_ds directly based on batch indices if needed,
        # but for this specific loop, let's adjust to what collate_fn provides.

        generated_ids = model.generate(
            pixel_values,
            max_new_tokens=64,
        )
        preds = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
        )

        # Decode labels to get ground truth texts
        # Replace -100 back to pad so decode doesn't crash, similar to compute_metrics
        labels[labels == -100] = processor.tokenizer.pad_token_id
        texts = processor.batch_decode(labels, skip_special_tokens=True)

        all_preds.extend(preds)
        all_targets.extend(texts)
        # Note: If filenames are crucial for per-sample analysis,
        # the collate_fn would need to be updated to return them in the batch.
        # For this fix, we are focusing on resolving the TypeError with images.
        # For simplicity, if collate_fn doesn't return filenames, we will omit them here
        # or assume they are not directly used in this specific loop structure.
        # However, the original loop expected fnames, so let's try to get them.
        # The original collate_fn only returns `inputs` which contains pixel_values and labels.
        # Let's temporarily remove `fnames` from the loop for now.
        # A more robust solution would be to modify the `collate_fn` to pass filenames.
        # For now, I'll remove `fnames` from the loop and handle it later if it becomes an issue.

print(f"Done. {len(all_preds)} predictions collected.")

100%|██████████| 25/25 [20:09<00:00, 48.37s/it]

Done. 800 predictions collected.


In [ ]:
# ── Metrics helpers ────────────────────────────────────────────────────

def cer(pred: str, ref: str) -> float:
    """Character Error Rate via edit distance (fallback if jiwer missing)."""
    ref = str(ref)
    pred = str(pred)
    if len(ref) == 0:
        return 0.0 if len(pred) == 0 else 1.0

    return editdistance.eval(pred, ref) / len(ref)


def compute_metrics(predictions, references):
    """
    Returns a dict with CER, WER, exact-match accuracy, and BLEU-4.

    - CER
    - Exact-match: case-sensitive full-string equality.
    """
    results = {}


    # ── CER ───────────────────────────────────────────────────────────────────
    if HAS_JIWER:
        cer_score = jiwer.cer(references, predictions)
    else:


        cer_score = float(np.mean([

            cer(p, r) for p, r in zip(predictions, references)
        ]))
    results["CER"] = round(cer_score * 100, 2)   # report as %

    # ── Exact match ───────────────────────────────────────────────────────────
    exact = sum(p == r for p, r in zip(predictions, references))
    results["Exact Match (%)"] = round(exact / len(references) * 100, 2)


    return results

# Evaluation results

### Explanation of Metrics

*   **CER (Character Error Rate):** This metric measures the number of character errors between the predicted text and the ground truth text, normalized by the length of the ground truth. **A lower CER value indicates better model performance**, meaning fewer character-level mistakes in the predictions.

*   **Exact Match (%):** This metric represents the percentage of predictions that perfectly match the ground truth text. **A higher Exact Match percentage indicates better model performance**, as it means more predictions are entirely correct.

In [ ]:

# ── Cell 3: Evaluate & display ────────────────────────────────────────────────

metrics = compute_metrics(all_preds, all_targets)

print("\n═══════════════════════════════")
print("        Evaluation Results      ")
print("═══════════════════════════════")
for k, v in metrics.items():
    print(f"  {k:<22} {v}")
print("═══════════════════════════════\n")


═══════════════════════════════
        Evaluation Results      
═══════════════════════════════
  CER                    67.1
  Exact Match (%)        0.0
═══════════════════════════════


In [ ]:
results_df = pd.DataFrame({
    "ground_truth": all_targets,
    "prediction": all_preds,
    "exact_match": [p == r for p, r in zip(all_preds, all_targets)],
    "cer"       : [cer(p, r) for p, r in zip(all_preds, all_targets)],
})

print(f"Overall exact-match rate : {results_df['exact_match'].mean()*100:.2f}%")
print(f"Mean CER (per-sample)    : {results_df['cer'].mean()*100:.2f}%\n")

# Worst 20 predictions by CER
print("── Top-20 LOWEST CER samples ──────────────────────────────────────────")
print(
    results_df.sort_values("cer", ascending=True)
              .head(20)
              .to_string(index=False)
)

Overall exact-match rate : 0.00%
Mean CER (per-sample)    : 67.10%

── Top-20 LOWEST CER samples ──────────────────────────────────────────
              ground_truth                 prediction  exact_match      cer
     Балта сабынан озбайды        Бапа сабынан озайды        False 0.142857
        кезде ойым Ержанға         жезде опым Ержанта        False 0.166667
        Қоршау жарым жылға           Коршаужарым жыла        False 0.166667
  Тбилисиден Дубайдан және     Тилисиден Драйдан жане        False 0.166667
     табиғи ресурстар және         табитиресурстаржне        False 0.190476
берген тапсырмаларын нақты   бергентапырмаларьн нокты        False 0.192308
      Сұмырай келсе құриды       Сктырай келсе кприды        False 0.200000
  Қолында үлкен қайшы адам   Кольнда улкен кайлы адам        False 0.208333
       оған баптың аталған         отан баптынаталган        False 0.210526
   апатына себепкер болған      матьна себелкер болан        False 0.217391
   орнынан қабылдау тура

# Evaluating FINE-TUNED model on the SAME test set and calculating metrics (CER and Exact Match)

In [ ]:
from torch.utils.data import DataLoader

all_preds   = []   # decoded strings
all_targets = []   # ground-truth strings
all_fnames  = []   # filenames (useful for error analysis)

print(f"Running inference on {len(test_ds)} test samples …")

Running inference on 800 test samples …


In [ ]:
with torch.no_grad():
    for batch in tqdm(test_loader):
        # Extract pixel_values, texts, and fnames from the collated batch
        pixel_values = batch["pixel_values"].to(loaded_model.device)
        labels = batch["labels"]
        # Assuming original OCRDataset's __getitem__ returns 'filename'
        # and collate_fn preserves it, or we need to pass filenames separately.
        # For now, let's assume `texts` in the loop corresponds to `labels` after decode.
        # And filenames need to be explicitly passed if collate_fn doesn't include them.
        # Since the original collate_fn doesn't explicitly return fnames,
        # we will extract from test_ds directly based on batch indices if needed,
        # but for this specific loop, let's adjust to what collate_fn provides.

        generated_ids = loaded_model.generate(
            pixel_values,
            max_new_tokens=64,
        )
        preds = loaded_processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
        )

        # Decode labels to get ground truth texts
        # Replace -100 back to pad so decode doesn't crash, similar to compute_metrics
        labels[labels == -100] = loaded_processor.tokenizer.pad_token_id
        texts = loaded_processor.batch_decode(labels, skip_special_tokens=True)

        all_preds.extend(preds)
        all_targets.extend(texts)
        # Note: If filenames are crucial for per-sample analysis,
        # the collate_fn would need to be updated to return them in the batch.
        # For this fix, we are focusing on resolving the TypeError with images.
        # For simplicity, if collate_fn doesn't return filenames, we will omit them here
        # or assume they are not directly used in this specific loop structure.
        # However, the original loop expected fnames, so let's try to get them.
        # The original collate_fn only returns `inputs` which contains pixel_values and labels.
        # Let's temporarily remove `fnames` from the loop for now.
        # A more robust solution would be to modify the `collate_fn` to pass filenames.
        # For now, I'll remove `fnames` from the loop and handle it later if it becomes an issue.

print(f"Done. {len(all_preds)} predictions collected.")

100%|██████████| 25/25 [56:01<00:00, 134.44s/it]

Done. 800 predictions collected.


# Evaluation results

In [ ]:

# ── Cell 3: Evaluate & display ────────────────────────────────────────────────

metrics = compute_metrics(all_preds, all_targets)

print("\n═══════════════════════════════")
print("        Evaluation Results      ")
print("═══════════════════════════════")
for k, v in metrics.items():
    print(f"  {k:<22} {v}")
print("═══════════════════════════════\n")


═══════════════════════════════
        Evaluation Results      
═══════════════════════════════
  CER                    3.7
  Exact Match (%)        48.62
═══════════════════════════════


In [ ]:
results_df = pd.DataFrame({
    "ground_truth": all_targets,
    "prediction": all_preds,
    "exact_match": [p == r for p, r in zip(all_preds, all_targets)],
    "cer"       : [cer(p, r) for p, r in zip(all_preds, all_targets)],
})

print(f"Overall exact-match rate : {results_df['exact_match'].mean()*100:.2f}%")
print(f"Mean CER (per-sample)    : {results_df['cer'].mean()*100:.2f}%\n")

# Worst 20 predictions by CER
print("── Top-20 LOWEST CER samples ──────────────────────────────────────────")
print(
    results_df.sort_values("cer", ascending=True)
              .head(20)
              .to_string(index=False)
)

Overall exact-match rate : 48.62%
Mean CER (per-sample)    : 3.70%

── Top-20 LOWEST CER samples ──────────────────────────────────────────
                              ground_truth                                 prediction  exact_match  cer
                    пиццаны шығарып жатқан                     пиццаны шығарып жатқан         True  0.0
                        қарай болуы мүмкін                         қарай болуы мүмкін         True  0.0
                    Нарықтағы шикізат қоры                     Нарықтағы шикізат қоры         True  0.0
           Жағажайда планермен ұшатын адам            Жағажайда планермен ұшатын адам         True  0.0
                     Локдаун және оқшаулау                      Локдаун және оқшаулау         True  0.0
       балық тағамдарынан пешке пісірілген        балық тағамдарынан пешке пісірілген         True  0.0
 тармақшасында көзделген жағдайда қойнауын  тармақшасында көзделген жағдайда қойнауын         True  0.0
                   жақсы ғыл

## 📊 Final Model Performance Comparison

In [ ]:
import pandas as pd

# Metrics for the model before fine-tuning (from cell mK51AmTUvHPf output)
model_metrics = {'CER': 67.1, 'Exact Match (%)': 0.0}

# Metrics for the loaded_model after fine-tuning (from cell RE2i8kB5K21m output)
loaded_model_metrics = {'CER': 3.7, 'Exact Match (%)': 48.62}

# Create a DataFrame for comparison
comparison_df = pd.DataFrame({
    'Metric': list(model_metrics.keys()),
    'Model (Before Fine-tuning)': list(model_metrics.values()),
    'Loaded Model (After Fine-tuning)': list(loaded_model_metrics.values())
})

display(comparison_df)


,Metric,Model (Before Fine-tuning),Loaded Model (After Fine-tuning)
0,CER,67.1,3.70
1,Exact Match (%),0.0,48.62
